# Task 3 — Intelligent Feature: Priority, SLA & Human-Review Routing

This notebook demonstrates the Task 3 addition to the Support Ticket Triage Classifier
(Tasks 1 & 2): a reasoning layer on top of the raw model prediction, plus honest evaluation
of where the classifier actually succeeds and fails on this small dataset.

**What's new in this notebook vs. Task 2:**
1. A more rigorous evaluation (5-fold cross-validation instead of a single train/test split)
2. Concrete, real failure cases pulled from the held-out set — with analysis of *why* they fail
3. Defensive error handling for malformed/empty/non-string input
4. The intelligent feature itself: confidence-gated human review + priority/SLA/routing/auto-response


In [1]:
from ticket_intelligence import (
    get_training_data, cross_validate_model, train_model, classify_ticket
)
import numpy as np

texts, labels = get_training_data()
print(f"Dataset: {len(texts)} labeled tickets across {len(set(labels))} categories")
from collections import Counter
Counter(labels)

Dataset: 58 labeled tickets across 4 categories


Counter({'General': 16, 'Billing': 14, 'Technical': 14, 'Account': 14})

## 1. Evaluation — cross-validated, not a single lucky split

A single train/test split on ~58 examples is noisy: the reported macro F1 can swing by
±0.2 depending on the random seed alone. Stratified 5-fold cross-validation gives a more
honest single number, since every example gets evaluated exactly once as part of a held-out fold.

In [2]:
cv = cross_validate_model(n_splits=5)
print("Per-fold macro F1:", np.round(cv["fold_scores"], 3))
print(f"Mean macro F1: {cv['mean_macro_f1']:.3f}  (+/- {cv['std_macro_f1']:.3f})")

Per-fold macro F1: [0.429 0.564 0.396 0.607 0.396]
Mean macro F1: 0.479  (+/- 0.089)


**Honest takeaway:** the cross-validated macro F1 lands around **0.45–0.55**, not the
0.85+ figure quoted in early project notes. With only ~14–16 labeled examples per category,
the TF-IDF vocabulary is too thin to separate categories that share support-ticket language
(e.g. "refund", "account", "not working"). This is a genuine, useful finding — it tells us
exactly what Task 4 needs to fix: more labeled data per category, not a fancier model.

## 2. A closer look: held-out predictions and real failure cases

In [3]:
model, ev = train_model()
print(f"Held-out split macro F1: {ev['macro_f1']:.3f}\n")
print(ev["report_text"])

Held-out split macro F1: 0.458

              precision    recall  f1-score   support

     Account       0.40      0.67      0.50         3
     Billing       0.50      0.50      0.50         4
     General       0.50      0.50      0.50         4
   Technical       0.50      0.25      0.33         4

    accuracy                           0.47        15
   macro avg       0.47      0.48      0.46        15
weighted avg       0.48      0.47      0.46        15



In [4]:
print(f"{'':4s} {'true':<10s} {'pred':<10s} ticket")
for text, true, pred in zip(ev["X_test"], ev["y_test"], ev["y_pred"]):
    mark = "OK  " if true == pred else "MISS"
    print(f"{mark} {true:<10s} {pred:<10s} {text}")

     true       pred       ticket
MISS Billing    Account    My invoice shows the wrong amount, please refund the difference
MISS General    Technical  How can I give feedback on a feature request?
OK   General    General    Do you have a referral or affiliate program?
MISS Billing    Account    I want to cancel my subscription and get a refund
OK   General    General    What languages does your platform support?
OK   Technical  Technical  I'm getting a 500 error when I try to log in
OK   Billing    Billing    Why is my bill higher than usual this cycle?
OK   Account    Account    I'm locked out after changing my phone number
OK   Billing    Billing    I never received an invoice for this month's charge
MISS Technical  Account    I can't connect the app to my calendar, sync keeps failing
MISS Technical  Billing    Export to PDF is broken, the file comes out empty
MISS Technical  General    API requests are timing out constantly today
MISS General    Billing    What's the difference bet

### Failure case analysis (real examples from the held-out set above)

| Ticket | True | Predicted | Why it likely failed |
|---|---|---|---|
| "My invoice shows the wrong amount, please refund the difference" | Billing | Account | "refund" alone isn't enough signal; the sentence never mentions charges/payment, so it reads as an account-correction request. |
| "I can't connect the app to my calendar, sync keeps failing" | Technical | Account | "connect"/"sync" overlaps with account-linking language seen in Account examples (SSO, merging accounts). |
| "API requests are timing out constantly today" | Technical | General | No training example uses "API" alongside a clear failure verb like "crash" or "error" — the model hasn't learned that "timing out" implies Technical. |
| "How do I downgrade my role from admin to member?" | Account | General | "How do I..." phrasing dominates the General category in training data, so the model leans General on any question-shaped ticket regardless of topic. |

**Pattern:** almost every miss is a case where the ticket's phrasing pattern (a question, a
generic verb) outweighs its topic-specific vocabulary. That's a data problem, not a model
problem — solvable with more, more varied labeled examples per category rather than a
different algorithm.

## 3. Error handling — the classifier should never crash on bad input

In [5]:
bad_inputs = [
    ("empty string", ""),
    ("whitespace only", "    "),
    ("None", None),
    ("wrong type (int)", 12345),
    ("too short", "hi"),
    ("very long input", "The app crashed. " * 200),
]

for label, value in bad_inputs:
    result = classify_ticket(model, value)
    print(f"{label:18s} -> {result}")

empty string       -> {'error': "Ticket text is empty. Please provide the ticket's content."}
whitespace only    -> {'error': "Ticket text is empty. Please provide the ticket's content."}
None               -> {'error': 'No ticket text was provided (received None).'}
wrong type (int)   -> {'error': 'Ticket text must be a string, got int.'}
too short          -> {'error': 'Ticket text is too short (2 chars) to classify reliably.'}
very long input    -> {'text': 'The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app

Every case above returns a structured result (an `error` key, or a `truncated` flag for
oversized input) instead of raising an exception — so a calling API or UI can display a
useful message instead of a stack trace.

## 4. The intelligent feature: priority, SLA, routing & human-review gating

On top of the raw category prediction, `classify_ticket()` adds a reasoning layer:
- **Priority** — from the category, escalated to High if urgency language is detected
- **SLA hours** — derived from priority
- **Routing team** — which internal team should own the ticket
- **needs_human_review** — `True` whenever model confidence falls below 0.40, so
  low-confidence predictions are routed to a person instead of auto-actioned
- **auto_response_draft** — a first-reply template to speed up the agent's response

In [6]:
demo_tickets = [
    "I was charged twice for my subscription this month",
    "The app crashes every time I upload a file, this is urgent",
    "I forgot my password and the reset email never arrives",
    "Do you have a referral program?",
]

for t in demo_tickets:
    r = classify_ticket(model, t)
    print(f'Ticket: "{t}"')
    print(f"  category={r['category']}  confidence={r['confidence']:.2f}  "
          f"needs_human_review={r['needs_human_review']}")
    print(f"  priority={r['priority']}  sla_hours={r['sla_hours']}  team={r['routing_team']}")
    print(f"  auto_response_draft: {r['auto_response_draft']}\n")

Ticket: "I was charged twice for my subscription this month"
  category=Billing  confidence=0.44  needs_human_review=False
  priority=Medium  sla_hours=24  team=Finance & Billing Ops
  auto_response_draft: Hi, thanks for reaching out about your billing concern. We're looking into the charge/invoice you mentioned and will follow up with a resolution shortly.

Ticket: "The app crashes every time I upload a file, this is urgent"
  category=Technical  confidence=0.37  needs_human_review=True
  priority=High  sla_hours=4  team=Platform Engineering
  auto_response_draft: Hi, sorry for the trouble you're running into. Our engineering team has been notified and is investigating the issue you described.

Ticket: "I forgot my password and the reset email never arrives"
  category=Account  confidence=0.43  needs_human_review=False
  priority=Medium  sla_hours=24  team=Identity & Access Support
  auto_response_draft: Hi, thanks for letting us know about your account issue. We're verifying your det

## 5. Summary

- Built a reasoning layer (priority, SLA, routing, human-review gating, auto-response) on
  top of the Task 2 classifier.
- Replaced the single noisy train/test split with 5-fold cross-validation for an honest
  headline metric (~0.5 macro F1 on this dataset size).
- Documented 4 concrete failure cases with root-cause analysis, all traced to limited
  training data rather than the modeling approach.
- Added defensive error handling for empty, malformed, wrong-type, and oversized input —
  verified with 6 explicit bad-input test cases, none of which raised an exception.
- **Next step (Task 4):** grow the labeled dataset per category — this is what the failure
  analysis says is actually needed to close the gap to a production-ready accuracy level.
